In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content ="Hello World",
    metadata={"source":"https:"}
)
sample_doc

Document(metadata={'source': 'https:'}, page_content='Hello World')

In [4]:
type(sample_doc)

langchain_core.documents.base.Document

In [5]:
#Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("Data/python.txt", encoding="utf-8")

In [6]:
document = loader.load()
document

[Document(metadata={'source': 'Data/python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [7]:
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader("Data/research2.pdf")

document = pdf_loader.load()
# document

# Ingestion Pipeline

In [8]:
import os

In [9]:
def load_all_pdfs():
    folder_path = "Data" ## Path of data
    total_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
    
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
    
            all_docs.extend(doc)
            total_docs += 1

    print("Total pdfs : ", total_docs)
    print("Total pages : ", len(all_docs))

    return all_docs
    

In [10]:
all_pdf_documents = load_all_pdfs()

Total pdfs :  2
Total pages :  32


## Chunks

In [ ]:
# !pip install langchain_text_splitters

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(document, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_doc = text_splitter.split_documents(document)

    return chunked_doc

In [23]:
chunks = split_doc(all_pdf_documents)
len(chunks)

321

## Embedding..

In [29]:
from sentence_transformers import SentenceTransformer

In [46]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("loading model.... ", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("Embedding dimentions = ", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, texts):
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print("Embeddings shape : ", embeddings.shape)
        return embeddings

In [47]:
embedding_manager = EmbeddingManager()

loading model....  all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimentions =  384


## Vector store

In [48]:
import chromadb
import uuid

In [57]:
class VectorStoreManager:
    def __init__(self, persist_directory="Data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        #create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        #create the collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata={"description": "Vector store collection for pdf embeddings in RAG"}
        )

        print("Initialized the vector stpre with collections: ", self.collection_name)
        print("Docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Num of documents does not match No. of embedings.")

        # STORE => ids, embeddings,, document, metadata
        ids = []
        all_metedata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata["content_length"] = len(doc.page_content)
            all_metedata.append(metadata)

            documents_content.append(doc.page_content)

            # embeddings_list.append(doc.page_content)
            embeddings_list.append(embedding)

        self.collection.add(
                ids = ids,
                metadatas= all_metedata,
                documents = documents_content,
                embeddings = embeddings_list
            )

        print("Total documents added in vector space = ", len(documents_content))
        print("docs in collection : ",self.collection.count())

In [58]:
vector_store = VectorStoreManager()

Initialized the vector stpre with collections:  pdf_documents
Docs in collection: 0


In [59]:
## data =>docs =>chunks => embeddings =>store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embeddings shape :  (321, 384)
Total documents added in vector space =  321
docs in collection :  321


# 2. Retrieval Pipeline

In [60]:
from sklearn.metrics.pairwise import cosine_similarity

In [81]:
class RAGRetrieval:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retreive(self, query, top_k=5, score_threshold=0.1):
        # Query => embedding..
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic seacrh
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        # Cosine similarity
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })
            print(f"retrieved {len(retrieved_docs)} documents")
        else:
            print("No documents found.")

        return retrieved_docs

In [82]:
rag_retriever = RAGRetrieval( embedding_manager, vector_store)

In [83]:
rag_retriever.retreive("what is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape :  (1, 384)
retrieved 4 documents


[{'id': 'doc_f8aee275-5144-4f14-b4ed-8ad3d29d969b',
  'document': 'layers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention',
  'metadata': {'date': '2017',
   'source': 'Data\\NIPS-2017-attention-is-all-you-need-Paper.pdf',
   'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)',
   'lastpage': '6008',
   'creationdate': '2026-03-30T16:55:00+00:00',
   'language': 'en-US',
   'producer': 'pdfcpu v0.9.1 dev',
   'title': 'Attention is All you Need',
   'doc_index': 18,
   'creator': 'PyPDF',
   'description-abstract': 'The dominant 